In [60]:
!pip install onnx onnxruntime mlflow

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.4/49.4 kB 826.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 41.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 102.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 147.8/147.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.3/86.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 912.3/912.3 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [67]:
import pandas as pd
import numpy as np
import json
import re

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score
from collections import Counter
import onnxruntime as ort
import mlflow.onnx

import onnx
from onnx import helper, TensorProto, numpy_helper

In [12]:
df_usda = pd.read_csv('comprehensive_foods_usda.csv')
df_usda.head()

,fdc_id,food_name,data_type,food_category,brand_owner,brand_name,ingredients,serving_size,serving_unit,household_serving,...,protein_g,saturated_fat_g,vitamin_c_mg,fiber_g,iron_mg,sodium_mg,sugar_g,cholesterol_mg,health_score,food_type
0,167782,"Abiyuch, raw",SR Legacy,Fruits and Fruit Juices,NaN,NaN,NaN,NaN,NaN,NaN,...,1.50,0.014,54.1,5.3,1.61,20.0,8.55,NaN,65,Fruits
1,171687,"Acerola juice, raw",SR Legacy,Fruits and Fruit Juices,NaN,NaN,NaN,NaN,NaN,NaN,...,0.40,0.068,1600.0,0.3,0.50,3.0,4.50,0.0,55,Fruits
2,171686,"Acerola, (west indian cherry), raw",SR Legacy,Fruits and Fruit Juices,NaN,NaN,NaN,NaN,NaN,NaN,...,0.40,0.068,1680.0,1.1,0.20,7.0,NaN,0.0,55,Fruits
3,168061,Acorn stew (Apache),SR Legacy,American Indian/Alaska Native Foods,NaN,NaN,NaN,NaN,NaN,NaN,...,6.81,1.280,0.0,0.7,1.00,130.0,0.34,20.0,50,Other
4,168992,"Agave, cooked (Southwest)",SR Legacy,American Indian/Alaska Native Foods,NaN,NaN,NaN,NaN,NaN,NaN,...,0.99,NaN,0.3,10.6,3.55,13.0,20.90,0.0,50,Other


In [13]:
df_healthy = pd.read_csv('healthy_foods_database.csv')
df_healthy.head()

,food_name,food_type,calories,protein_g,fat_g,carbs_g,fiber_g,sugar_g,sodium_mg,health_score
0,"Abiyuch, raw",Fruits,69.0,1.50,0.10,17.60,5.30,8.55,20.000,65
1,"Agave, raw (Southwest)",Other,68.0,0.52,0.15,16.20,6.60,2.58,14.000,60
2,"Almond butter, creamy",Other,603.0,20.80,53.00,21.20,9.72,NaN,0.996,70
3,"Amaranth grain, uncooked",Grains,371.0,13.60,7.02,65.20,6.70,1.69,4.000,70
4,"APPLEBEE'S, chili",Other,656.0,12.60,9.79,4.57,1.40,2.27,381.000,60


In [14]:
# df_allergens = pd.read_csv('foods_allergens.csv')
# df_allergens.head()

In [15]:
# df_dietary = pd.read_csv('foods_dietary_restrictions.csv')
# df_dietary.head()

In [16]:
print(len(df_usda))
print(len(df_healthy))
# print(len(df_allergens))
# print(len(df_dietary))

40000
9028


In [17]:
usda_names = set(df_usda['food_name'].dropna().astype(str))
healthy_names = set(df_healthy['food_name'].dropna().astype(str))
# allergen_names = set(df_allergens['product_name'].dropna().astype(str))
# dietary_names = set(df_dietary['product_name'].dropna().astype(str))

print(f"Overlap analysis:")
print(f"USDA dan Healthy\t\t: {len(usda_names & healthy_names):>6,}")
# print(f"USDA dan Allergens\t\t: {len(usda_names & allergen_names):>6,}")
# print(f"USDA dan Dietary\t\t: {len(usda_names & dietary_names):>6,}")
# print(f"Allergens dan Dietary\t\t: {len(allergen_names & dietary_names):>6,}")

Overlap analysis:
USDA dan Healthy		:  7,572


In [18]:
df_usda.isnull().sum()

,0
fdc_id,0
food_name,0
data_type,0
food_category,46
brand_owner,8617
brand_name,9746
ingredients,8344
serving_size,8158
serving_unit,8158
household_serving,9210


In [19]:
df_usda['food_name'].duplicated().sum()

np.int64(9100)

In [20]:
print("Distribusi food_type:")
print(df_usda['food_type'].value_counts())
print()
print(df_usda['health_score'].describe())

Distribusi food_type:
food_type
Other              15521
Snacks & Sweets     4971
Dairy               4756
Fruits              4566
Meat & Poultry      4211
Vegetables          2362
Grains              2010
Beverages            890
Seafood              713
Name: count, dtype: int64

count    40000.000000
mean        48.887250
std          9.850912
min         20.000000
25%         40.000000
50%         50.000000
75%         55.000000
max         75.000000
Name: health_score, dtype: float64


In [21]:
NUTRITION_COLS = ['calories','carbs_g','calcium_mg','fat_g','protein_g',
                  'saturated_fat_g','vitamin_c_mg','fiber_g','iron_mg',
                  'sodium_mg','sugar_g','cholesterol_mg']

master = df_usda[['fdc_id','food_name','food_category','food_type',
                  'health_score','ingredients'] + NUTRITION_COLS].copy()

master = master.drop_duplicates(subset=['food_name']).reset_index(drop=True)
master['is_healthy_subset'] = master['food_name'].isin(healthy_names)

In [22]:
(df_usda[NUTRITION_COLS] < 0).any()

,0
calories,False
carbs_g,True
calcium_mg,False
fat_g,False
protein_g,False
saturated_fat_g,False
vitamin_c_mg,False
fiber_g,False
iron_mg,False
sodium_mg,False


In [23]:
print(master[master['carbs_g'] < 0][['food_name', 'carbs_g']])

                                        food_name  carbs_g
1899                           Bison, ground, raw  -0.1500
2716          Chicken, breast, meat and skin, raw  -0.4280
2875       Chicken, drumstick, meat and skin, raw  -0.4750
2921           Chicken, thigh, meat and skin, raw  -0.1730
2926            Chicken, wing, meat and skin, raw  -0.4590
4124                 Halibut, frozen, wild caught  -0.0602
5837                  Pork, belly, with skin, raw  -0.7050
5838                  Pork, chop, center cut, raw  -0.5620
7591  Tuna, ahi or yellowfin, frozen, wild caught  -0.1040


In [24]:
for col in NUTRITION_COLS:
  master[col] = master.groupby('food_type')[col].transform(lambda x: x.fillna(x.median()))
  master[col] = master[col].clip(lower=0)


In [25]:
master[NUTRITION_COLS].isnull().sum()

,0
calories,0
carbs_g,0
calcium_mg,0
fat_g,0
protein_g,0
saturated_fat_g,0
vitamin_c_mg,0
fiber_g,0
iron_mg,0
sodium_mg,0


In [26]:
ALLERGEN_KEYWORDS = {
    'gluten': ['wheat','barley','rye','malt','semolina','spelt','farro','bulgur',
               'triticale','seitan','gluten','flour'],
    'dairy' : ['milk','cheese','butter','cream','yogurt','whey','casein','lactose',
               'ghee','curd','milkfat','dairy'],
    'nuts'  : ['almond','cashew','walnut','pecan','pistachio','hazelnut','macadamia',
               'brazil nut','peanut','peanuts'],
    'soy'   : ['soy','soya','soybean','tofu','edamame','tempeh','miso'],
    'eggs'  : ['egg','eggs','albumin','ovalbumin','meringue'],
    'fish'  : ['fish','salmon','tuna','cod','tilapia','anchovy','sardine','shellfish',
               'shrimp','crab','lobster','oyster','clam','scallop'],
}

MEAT_KEYWORDS = ['beef','pork','chicken','turkey','lamb','duck','bacon','sausage',
                 'pepperoni','ham','salami','venison','goat','meat','gelatin',
                 'lard','tallow','rennet','chorizo','prosciutto','meatball']

In [27]:
def detect_allergen(text_fields, keywords):
  """text_fields: list of strings (ingredients + food_name). Return True kalau ada keyword."""
  combined = ' '.join([str(t) for t in text_fields if not pd.isna(t) and t != '']).upper()
  if not combined:
      return False
  for kw in keywords:
      if re.search(r'\b' + re.escape(kw.upper()) + r'\b', combined):
          return True
  return False

In [28]:
for allergen, kws in ALLERGEN_KEYWORDS.items():
  master[f'contains_{allergen}'] = master.apply(
      lambda r: detect_allergen([r['ingredients'], r['food_name']], kws), axis=1
  )

master['contains_meat'] = master.apply(
    lambda r: detect_allergen([r['ingredients'], r['food_name']], MEAT_KEYWORDS), axis=1
)

In [29]:
for a in ALLERGEN_KEYWORDS:
    col = f'contains_{a}'
    print(f"{col:20s}: {master[col].sum():>6,}")
print(f"{'contains_meat':20s}: {master['contains_meat'].sum():>6,}")

contains_gluten     :  8,042
contains_dairy      :  9,232
contains_nuts       :  2,093
contains_soy        :  6,605
contains_eggs       :  2,834
contains_fish       :  1,284
contains_meat       :  6,608


In [30]:
master[master['contains_dairy'] == True][['food_name', 'ingredients']]

,food_name,ingredients
7,"Agutuk, fish with shortening (Alaskan ice crea...",NaN
8,"Agutuk, fish/berry with seal oil (Alaskan ice ...",NaN
9,"Agutuk, meat-caribou (Alaskan ice cream) (Alas...",NaN
28,"Alcoholic beverage, liqueur, coffee with cream...",NaN
78,"Almond butter, creamy",NaN
...,...,...
30886,"BEAN & CHEESE TRADITIONAL BURRITO, BEAN & CHEESE","FILLING: WATER, CHEDDAR CHEESE, PINTO BEANS, G..."
30887,"BEAN & CHEESE WITH PINTO BEANS, REAL CHEDDAR C...","FILLING: WATER, PINTO BEANS, CHEDDAR CHEESE (P..."
30888,BEAN & CHEESE WRAP,"TORTILLA (ORGANIC UNBLEACHED FLOUR, WATER, NON..."
30889,"BEAN & CHEESE XX LARGE BURRITO, BEAN & CHEESE","WATER WHEAT FLOUR (ENRICHED WITH NIACIN, REDUC..."


In [31]:
master['food_type'].value_counts()

,count
food_type,
Other,12181
Meat & Poultry,3923
Snacks & Sweets,3758
Dairy,3253
Fruits,3085
Vegetables,1674
Grains,1573
Beverages,835
Seafood,618


In [32]:
veg_excluded_types   = ['Meat & Poultry', 'Seafood']
vegan_excluded_types = ['Meat & Poultry', 'Seafood', 'Dairy']

master['is_vegetarian'] = (~master['food_type'].isin(veg_excluded_types)) & (~master['contains_fish']) & (~master['contains_meat'])
master['is_vegan'] = (~master['food_type'].isin(vegan_excluded_types)) & (~master['contains_dairy']) & (~master['contains_eggs']) & \
                      (~master['contains_fish']) & (~master['contains_meat'])

master.loc[master['food_type'] == 'Dairy',   'contains_dairy'] = True
master.loc[master['food_type'] == 'Seafood', 'contains_fish']  = True

In [33]:
print(f"is_vegetarian: {master['is_vegetarian'].sum():,}")
print(f"is_vegan     : {master['is_vegan'].sum():,}")

is_vegetarian: 22,883
is_vegan     : 14,825


In [34]:
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(master[NUTRITION_COLS].values).astype(np.float32)

norms = np.linalg.norm(features_scaled, axis=1, keepdims=True)
norms[norms == 0] = 1.0
features_normalized = features_scaled / norms

In [35]:
N, M = features_normalized.shape

In [36]:

x_input = helper.make_tensor_value_info('target_nutrition', TensorProto.FLOAT, [M])
sim_output = helper.make_tensor_value_info('similarities', TensorProto.FLOAT, [N])

# konstanta yang di-embed
init_scale = numpy_helper.from_array(scaler.scale_.astype(np.float32), name='scaler_scale')
init_min = numpy_helper.from_array(scaler.min_.astype(np.float32), name='scaler_min')
init_feat = numpy_helper.from_array(features_normalized, name='feature_matrix_normalized')
init_eps = numpy_helper.from_array(np.array([1e-12], dtype=np.float32), name='eps')
init_axes = numpy_helper.from_array(np.array([0], dtype=np.int64), name='reduce_axes')

In [37]:
nodes = [
    # Step 1: Apply MinMaxScaler
    # x_scaled = nutrition_vector * scaler_scale + scaler_min
    helper.make_node('Mul', ['target_nutrition', 'scaler_scale'], ['x_mul']),
    helper.make_node('Add', ['x_mul', 'scaler_min'], ['x_scaled']),

    # Step 2: L2 normalize
    # x_norm = ||x_scaled||_2
    helper.make_node('ReduceL2', ['x_scaled', 'reduce_axes'], ['x_norm'], keepdims=1),
    # x_norm_safe = x_norm + eps (avoid division by zero)
    helper.make_node('Add', ['x_norm', 'eps'], ['x_norm_safe']),
    # x_normalized = x_scaled / x_norm_safe
    helper.make_node('Div', ['x_scaled', 'x_norm_safe'], ['x_normalized']),
    # similarities = feature_matrix_normalized @ x_normalized
    helper.make_node('MatMul', ['feature_matrix_normalized', 'x_normalized'], ['similarities']),
]

In [38]:
graph = helper.make_graph(
    nodes=nodes,
    name='FoodRecommendation',
    inputs=[x_input],
    outputs=[sim_output],
    initializer=[init_scale, init_min, init_feat, init_eps, init_axes]
)

model = helper.make_model(
    graph,
    producer_name='food-recommender',
    opset_imports=[helper.make_opsetid('', 18)]
)

In [39]:
model.ir_version = 9

In [40]:
onnx.save(model, 'food_recommender.onnx')

In [41]:
onnx.checker.check_model('food_recommender.onnx')

In [42]:
export_cols = ['food_name','food_category','food_type','health_score','is_healthy_subset',
               'is_vegetarian','is_vegan'] + [f'contains_{a}' for a in ALLERGEN_KEYWORDS] + ['contains_meat'] + NUTRITION_COLS

In [43]:
foods_export = master[export_cols].copy()
foods_export.insert(0, 'id', range(len(foods_export)))
foods_export = foods_export.rename(columns={
    'food_name':'name', 'food_type':'type', 'food_category':'category'
})

bool_cols = ['is_healthy_subset','is_vegetarian','is_vegan'] + [f'contains_{a}' for a in ALLERGEN_KEYWORDS] + ['contains_meat']
for col in bool_cols:
    foods_export[col] = foods_export[col].astype(int)

foods_export.to_json('foods.json', orient='records')

In [44]:
metadata = {
    'model': {
        'file': 'food_recommender.onnx',
        'input_name': 'target_nutrition',
        'output_name': 'similarities',
        'feature_order': NUTRITION_COLS,
        'n_foods': len(master),
        'n_features': len(NUTRITION_COLS),
    },
    'data_file': 'foods.json',
    'allergens': list(ALLERGEN_KEYWORDS.keys()),
    'diet_options': ['none', 'vegetarian', 'vegan'],
    'goals': ['lose_weight', 'maintain', 'gain_muscle'],
    'activity_levels': {
        'sedentary': 1.2, 'light': 1.375, 'moderate': 1.55,
        'active': 1.725, 'very_active': 1.9
    },
    'goal_calorie_adjustment': {
        'lose_weight': -500, 'maintain': 0, 'gain_muscle': 300
    },
    'macro_ratios': {
        'lose_weight': {'protein':0.30, 'carbs':0.40, 'fat':0.30},
        'gain_muscle': {'protein':0.30, 'carbs':0.50, 'fat':0.20},
        'maintain'   : {'protein':0.20, 'carbs':0.50, 'fat':0.30},
    },
    'bmi_categories': {
        'underweight': [0, 18.5], 'normal': [18.5, 25],
        'overweight': [25, 30], 'obese': [30, 100]
    },
    'scoring_weights': {
        'similarity': 0.6, 'health_score': 0.3, 'healthy_subset_bonus': 0.1
    }
}

In [45]:
with open('metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

# Evaluation

In [55]:
master['food_type'].value_counts()

,count
food_type,
Other,12181
Meat & Poultry,3923
Snacks & Sweets,3758
Dairy,3253
Fruits,3085
Vegetables,1674
Grains,1573
Beverages,835
Seafood,618


In [56]:
sess = ort.InferenceSession('food_recommender.onnx')
in_name = sess.get_inputs()[0].name

X_raw  = master[NUTRITION_COLS].values.astype(np.float32)
labels = master['food_type'].values
cat_counts = Counter(labels)

Ks = [1, 3, 5]
res = {K: {'p': []} for K in Ks}

for i in range(len(X_raw)):
    sims = sess.run(None, {in_name: X_raw[i]})[0]
    sims[i] = -np.inf
    order = np.argsort(-sims)
    q = labels[i]
    for K in Ks:
        hits = np.sum(labels[order[:K]] == q)
        res[K]['p'].append(hits / K)

print(f"{'K':>4} | {'Precision@K':>12} |")
for K in Ks:
    print(f"{K:>4} | {np.mean(res[K]['p']):>12.3f} |")

   K |  Precision@K |
   1 |        0.672 |
   3 |        0.626 |
   5 |        0.604 |


In [59]:
master['is_healthy'] = (master['health_score'] >= 60).astype(int)

X_raw  = master[NUTRITION_COLS].values.astype(np.float32)
labels = master['is_healthy'].values
K = 5

y_true, y_pred = [], []

for i in range(len(X_raw)):
    sims = sess.run(None, {in_name: X_raw[i]})[0]
    sims[i] = -np.inf
    topk = np.argpartition(-sims, K)[:K]
    vote = np.bincount(labels[topk]).argmax()
    y_pred.append(vote)
    y_true.append(labels[i])

print(classification_report(y_true, y_pred, target_names=['Not Healthy','Healthy']))
print(confusion_matrix(y_true, y_pred))

              precision    recall  f1-score   support

 Not Healthy       0.93      0.93      0.93     23420
     Healthy       0.78      0.80      0.79      7480

    accuracy                           0.90     30900
   macro avg       0.86      0.86      0.86     30900
weighted avg       0.90      0.90      0.90     30900

[[21696  1724]
 [ 1511  5969]]


# MLFlow

In [68]:
onnx_model = onnx.load('food_recommender.onnx')
mlflow.set_experiment("Food_Recommender_System")

with mlflow.start_run() as run:
  mlflow.log_param("K_neighbors", K)

  acc = accuracy_score(y_true, y_pred)
  prec = precision_score(y_true, y_pred)

  mlflow.log_metric("accuracy", acc)
  mlflow.log_metric("precision", prec)

  mlflow.onnx.log_model(onnx_model=onnx_model, name="onnx_model")

  print(run.info.run_id)

aa025368773845989271e88fc0de5167


In [69]:
mlflow.search_runs(experiment_ids='1')

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.precision,metrics.accuracy,params.K_neighbors,tags.mlflow.source.name,tags.mlflow.user,tags.mlflow.runName,tags.mlflow.source.type
0,aa025368773845989271e88fc0de5167,1,FINISHED,/content/mlruns/1/aa025368773845989271e88fc0de...,2026-06-10 00:00:42.096000+00:00,2026-06-10 00:00:49.589000+00:00,0.7759,0.895307,5,fileId=1wBGnS9b3EechJZVZD2knOoFWRxN6SBNK,root,rumbling-deer-414,NOTEBOOK
1,5c0e6e5b18df4a0990a237f7c3300fbb,1,FINISHED,/content/mlruns/1/5c0e6e5b18df4a0990a237f7c330...,2026-06-09 23:48:31.633000+00:00,2026-06-09 23:48:37.343000+00:00,NaN,0.895307,5,fileId=1wBGnS9b3EechJZVZD2knOoFWRxN6SBNK,root,gentle-grub-861,NOTEBOOK
2,32bbca2337b34b36a45e6c4d84d45394,1,FINISHED,/content/mlruns/1/32bbca2337b34b36a45e6c4d84d4...,2026-06-09 23:48:14.145000+00:00,2026-06-09 23:48:22.666000+00:00,NaN,0.895307,5,fileId=1wBGnS9b3EechJZVZD2knOoFWRxN6SBNK,root,chill-quail-169,NOTEBOOK
